In [0]:
from pyspark.sql import functions as F

train_df = spark.read.table("workspace.ecommerce.train_dataset")
test_df  = spark.read.table("workspace.ecommerce.test_dataset")

print("Train count:", train_df.count())
print("Test count:", test_df.count())

In [0]:
label_counts = train_df.groupBy("label").count().collect()

count_0 = [r['count'] for r in label_counts if r['label'] == 0][0]
count_1 = [r['count'] for r in label_counts if r['label'] == 1][0]

weight_0 = (count_0 + count_1) / (2 * count_0)
weight_1 = (count_0 + count_1) / (2 * count_1)

train_df = train_df.withColumn(
    "classWeight",
    F.when(F.col("label") == 1, weight_1).otherwise(weight_0)
)

print("Class weights added")

In [0]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "total_events",
    "unique_products",
    "avg_price"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

train_ml = assembler.transform(train_df).select("features", "label", "classWeight")
test_ml  = assembler.transform(test_df).select("features", "label")

print("Feature vector created")

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

In [0]:
from pyspark.ml.classification import LogisticRegression

lr_results = {}

print("\nTraining Logistic Regression")

for reg in [0.0, 0.01, 0.1]:
    lr = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        weightCol="classWeight",
        regParam=reg,
        maxIter=20
    )
    
    model = lr.fit(train_ml)
    pred = model.transform(test_ml)
    auc = evaluator.evaluate(pred)
    
    lr_results[reg] = auc
    print(f"LR regParam={reg} → AUC={auc}")

best_lr_param = max(lr_results, key=lr_results.get)
best_lr_auc = lr_results[best_lr_param]

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf_results = {}

print("\nTraining RandomForest")

for trees in [20, 50]:
    for depth in [5, 10]:
        
        rf = RandomForestClassifier(
            featuresCol="features",
            labelCol="label",
            numTrees=trees,
            maxDepth=depth,
            seed=42
        )
        
        model = rf.fit(train_ml)
        pred = model.transform(test_ml)
        auc = evaluator.evaluate(pred)
        
        rf_results[(trees, depth)] = auc
        print(f"RF trees={trees}, depth={depth} → AUC={auc}")

best_rf_param = max(rf_results, key=rf_results.get)
best_rf_auc = rf_results[best_rf_param]

In [0]:
print("\n====== FINAL MODEL COMPARISON ======")

print(f"Best Logistic AUC: {best_lr_auc} (regParam={best_lr_param})")
print(f"Best RandomForest AUC: {best_rf_auc} (trees={best_rf_param[0]}, depth={best_rf_param[1]})")

best_model = "RandomForest" if best_rf_auc > best_lr_auc else "LogisticRegression"

print("Best Model:", best_model)